In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
xls = pd.ExcelFile('/content/drive/MyDrive/warfarin.xls')
df_subject_data = xls.parse('Subject Data')
print("Loaded:", len(df_subject_data), "rows")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: 5700 rows


In [3]:
df = df_subject_data.copy()
df = df[df['Subject Reached Stable Dose of Warfarin'] == 1]
df = df[df['Therapeutic Dose of Warfarin'].notna()]
df = df[df['Therapeutic Dose of Warfarin'] < 300]
df = df[df['CYP2C9 consensus'].notna()]
df = df[df['VKORC1 -1639 consensus'].notna()]
df = df[df['Height (cm)'].notna()]
df = df[df['Weight (kg)'].notna()]
df = df[df['Age'].notna()]
print("Remaining patients:", len(df))

Remaining patients: 3449


In [4]:
def age_to_ordinal(age_band):
    try:
        lower = int(str(age_band).split('-')[0].strip().split('+')[0])
        return lower // 10
    except:
        return np.nan
df['age_ordinal'] = df['Age'].apply(age_to_ordinal)
df = df[df['age_ordinal'].notna()]
print("After age conversion:", len(df))

After age conversion: 3449


In [5]:
inducer_cols = ['Carbamazepine (Tegretol)', 'Phenytoin (Dilantin)', 'Rifampin or Rifampicin']
df[inducer_cols] = df[inducer_cols].fillna(0)
df['enzyme_inducer'] = (df[inducer_cols].sum(axis=1) > 0).astype(int)
df['Amiodarone (Cordarone)'] = df['Amiodarone (Cordarone)'].fillna(0)
print("Enzyme inducer flag done.")

Enzyme inducer flag done.


In [6]:
race_omb = df['Race (OMB)'].astype(str).str.strip().str.lower()
asian_mask = race_omb == 'asian'
df_asian = df[asian_mask].copy()
df_main  = df[~asian_mask].copy()
print("Asian held-out subgroup:", len(df_asian))
print("Main cohort for train/test:", len(df_main))

raw = df_subject_data.copy()
raw_indian = (raw['Race (Reported)'].astype(str).str.strip() == 'Indian')
n_indian = raw_indian.sum()
n_vkorc1 = (raw_indian & raw['VKORC1 -1639 consensus'].notna()).sum()
print(f"Indian patients: {n_indian} | with complete VKORC1: {n_vkorc1}")

Asian held-out subgroup: 1120
Main cohort for train/test: 2329
Indian patients: 43 | with complete VKORC1: 3


In [7]:
from sklearn.model_selection import train_test_split
predictors = ['age_ordinal', 'Height (cm)', 'Weight (kg)',
              'CYP2C9 consensus', 'VKORC1 -1639 consensus',
              'Race (OMB)', 'enzyme_inducer', 'Amiodarone (Cordarone)']
def build_X(data):
    return pd.get_dummies(data[predictors],
                          columns=['CYP2C9 consensus', 'VKORC1 -1639 consensus', 'Race (OMB)'])
X_main = build_X(df_main)
y_main = np.sqrt(df_main['Therapeutic Dose of Warfarin'])
X_train, X_test, y_train, y_test = train_test_split(
    X_main, y_main, test_size=0.3, random_state=42)
X_asian = build_X(df_asian).reindex(columns=X_train.columns, fill_value=0)
y_asian = np.sqrt(df_asian['Therapeutic Dose of Warfarin'])
print("Train:", X_train.shape, "| Test:", X_test.shape, "| Asian:", X_asian.shape)

Train: (1630, 20) | Test: (699, 20) | Asian: (1120, 20)


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
lr = LinearRegression().fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=500, random_state=42).fit(X_train, y_train)
print("Both models trained.")

Both models trained.


In [9]:
from sklearn.metrics import mean_absolute_error
def evaluate(model, X, y_sqrt, label):
    pred = model.predict(X) ** 2
    actual = y_sqrt.values ** 2
    mae = mean_absolute_error(actual, pred)
    pct20 = np.mean(np.abs(actual - pred) <= 0.2 * actual) * 100
    print(f"{label}: n={len(actual)}, MAE={mae:.2f} mg/week, within 20%={pct20:.1f}%")

evaluate(lr, X_test, y_test, "Linear regression - main test")
evaluate(rf, X_test, y_test, "Random forest    - main test")
print()
evaluate(lr, X_asian, y_asian, "Linear regression - Asian subgroup")
evaluate(rf, X_asian, y_asian, "Random forest    - Asian subgroup")

Linear regression - main test: n=699, MAE=9.43 mg/week, within 20%=47.2%
Random forest    - main test: n=699, MAE=10.24 mg/week, within 20%=42.9%

Linear regression - Asian subgroup: n=1120, MAE=6.70 mg/week, within 20%=40.7%
Random forest    - Asian subgroup: n=1120, MAE=7.24 mg/week, within 20%=38.1%


In [10]:
from sklearn.model_selection import KFold

# 5-fold cross-validation on main cohort
X_all = build_X(df_main).reset_index(drop=True)
y_all = np.sqrt(df_main['Therapeutic Dose of Warfarin']).reset_index(drop=True)
dose_all = df_main['Therapeutic Dose of Warfarin'].values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_scores(model_ctor):
    maes, pcts = [], []
    for tr, te in kf.split(X_all):
        m = model_ctor().fit(X_all.iloc[tr], y_all.iloc[tr])
        pred = m.predict(X_all.iloc[te]) ** 2
        actual = dose_all[te]
        maes.append(mean_absolute_error(actual, pred))
        pcts.append(np.mean(np.abs(actual - pred) <= 0.2 * actual) * 100)
    return np.array(maes), np.array(pcts)

lr_mae, lr_pct = cv_scores(lambda: LinearRegression())
rf_mae, rf_pct = cv_scores(lambda: RandomForestRegressor(n_estimators=500, random_state=42))
print("=== 5-FOLD CROSS-VALIDATION ===")
print(f"Linear: MAE {lr_mae.mean():.2f} ± {lr_mae.std():.2f} | within20% {lr_pct.mean():.1f} ± {lr_pct.std():.1f}")
print(f"RandomF: MAE {rf_mae.mean():.2f} ± {rf_mae.std():.2f} | within20% {rf_pct.mean():.1f} ± {rf_pct.std():.1f}")

# Bootstrap 95% CIs
def bootstrap_ci(model, X, actual_dose, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    pred = model.predict(X) ** 2
    n = len(actual_dose)
    maes = [mean_absolute_error(actual_dose[idx], pred[idx])
            for idx in (rng.integers(0, n, n) for _ in range(n_boot))]
    point = mean_absolute_error(actual_dose, pred)
    lo, hi = np.percentile(maes, [2.5, 97.5])
    return point, lo, hi

print("\n=== BOOTSTRAP 95% CONFIDENCE INTERVALS ===")
for m, X, y, lbl in [(lr, X_test, y_test, "Linear  main test"),
                     (rf, X_test, y_test, "RandomF main test"),
                     (lr, X_asian, y_asian, "Linear  Asian"),
                     (rf, X_asian, y_asian, "RandomF Asian")]:
    p, lo, hi = bootstrap_ci(m, X, (y.values**2))
    print(f"{lbl}: MAE {p:.2f} (95% CI {lo:.2f}-{hi:.2f})")

=== 5-FOLD CROSS-VALIDATION ===
Linear: MAE 9.05 ± 0.37 | within20% 48.4 ± 2.0
RandomF: MAE 9.81 ± 0.22 | within20% 44.2 ± 1.5

=== BOOTSTRAP 95% CONFIDENCE INTERVALS ===
Linear  main test: MAE 9.43 (95% CI 8.68-10.31)
RandomF main test: MAE 10.24 (95% CI 9.45-11.14)
Linear  Asian: MAE 6.70 (95% CI 6.32-7.06)
RandomF Asian: MAE 7.24 (95% CI 6.88-7.61)


In [11]:
pred_norace = ['age_ordinal', 'Height (cm)', 'Weight (kg)',
               'CYP2C9 consensus', 'VKORC1 -1639 consensus',
               'enzyme_inducer', 'Amiodarone (Cordarone)']
def build_X_norace(data):
    return pd.get_dummies(data[pred_norace],
                          columns=['CYP2C9 consensus', 'VKORC1 -1639 consensus'])

def eval_asian(build_fn, label):
    Xall = build_fn(df_main)
    yall = np.sqrt(df_main['Therapeutic Dose of Warfarin'])
    Xtr, _, ytr, _ = train_test_split(Xall, yall, test_size=0.3, random_state=42)
    Xas = build_fn(df_asian).reindex(columns=Xtr.columns, fill_value=0)
    yas = df_asian['Therapeutic Dose of Warfarin'].values
    for name, ctor in [("Linear", lambda: LinearRegression()),
                       ("RandomF", lambda: RandomForestRegressor(n_estimators=500, random_state=42))]:
        m = ctor().fit(Xtr, ytr)
        pred = m.predict(Xas) ** 2
        mae = mean_absolute_error(yas, pred)
        pct = np.mean(np.abs(yas - pred) <= 0.2 * yas) * 100
        print(f"[{label:9}] {name}: Asian MAE {mae:.2f} mg/week, within20% {pct:.1f}%")

print("=== TRANSPORTABILITY: WITH vs WITHOUT RACE ===")
eval_asian(build_X,        "with race")
eval_asian(build_X_norace, "no race")

=== TRANSPORTABILITY: WITH vs WITHOUT RACE ===
[with race] Linear: Asian MAE 6.70 mg/week, within20% 40.7%
[with race] RandomF: Asian MAE 7.24 mg/week, within20% 38.1%
[no race  ] Linear: Asian MAE 6.57 mg/week, within20% 41.7%
[no race  ] RandomF: Asian MAE 7.30 mg/week, within20% 38.1%


In [12]:
import sys, sklearn
print("Python", sys.version.split()[0])
print("scikit-learn", sklearn.__version__)
print("pandas", pd.__version__)
print("numpy", np.__version__)

Python 3.13.15
scikit-learn 1.6.1
pandas 2.2.3
numpy 2.1.3
